# Atelier ML - Analyse d'image : utilisation du modèle
## Utilisation du modèle avec de nouvelles images
On utilise le modèle entraîné dans `classification.ipynb` pour classer de **nouvelles images** de vêtements trouvées sur Internet.

- Il faut avoir exécuté `classification.ipynb` avant ce notebook : le répertoire `modele/` doit contenir `modele.json` et `modele.weights.h5`.
- Les images à tester sont à placer dans le répertoire `images/` (de préférence des vêtements photographiés sur un fond clair et uni).

## Chargement du modèle
On charge la **description** du modèle (JSON), puis ses **poids**. Pour utiliser le modèle à 4 convolutions (partie optionnelle), remplacer `NOM_MODELE` par `'modele_4_convolutions'`.

In [ ]:
import os

import numpy as np
from matplotlib import pyplot as plt
from PIL import Image, ImageFilter
from keras.models import model_from_json

# Nom du modèle à utiliser : 'modele' (1 convolution) ou 'modele_4_convolutions' (optionnel)
NOM_MODELE = 'modele'

# Chargement de la description du modèle
fichier_json = open(f'modele/{NOM_MODELE}.json', 'r')
modele_json = fichier_json.read()
fichier_json.close()

# Chargement de la description des poids du modèle
modele = model_from_json(modele_json)

# Chargement des poids
modele.load_weights(f"modele/{NOM_MODELE}.weights.h5")

print("Modèle chargé !")

Modèle chargé !


## Catégories de classification

In [4]:
# Définition des catégories de classifications
classes = ["Un T-shirt/haut",
           "Un pantalon",
           "Un pull",
           "Une robe",
           "Un manteau",
           "Des sandales",
           "Une chemise",
           "Des baskets",
           "Un sac",
           "Des bottes de cheville"]

In [5]:
import numpy as np
from PIL import Image, ImageFilter


def load_and_preprocess_image(image_path, target_size=28, content_size=20):
    """
    Load an image, convert to grayscale, and preprocess for model input.
    
    Args:
        image_path: Path to the input image
        target_size: Size of the output square image (default: 28x28)
        content_size: Size of the resized content within the target (default: 20)
    
    Returns:
        numpy.ndarray: Preprocessed image array ready for model prediction
    """
    image = Image.open(image_path).convert('L')
    resized_img = resize_with_aspect_ratio(image, content_size)
    canvas = create_centered_image(resized_img, target_size)
    normalized_array = normalize_pixels(canvas)
    
    return normalized_array.reshape(1, target_size, target_size, 1)


def resize_with_aspect_ratio(image, target_dimension):
    """
    Resize image maintaining aspect ratio.
    
    Args:
        image: PIL Image object
        target_dimension: Target size for the longest dimension
    
    Returns:
        PIL Image: Resized and sharpened image
    """
    width, height = image.size
    
    if width > height:
        new_width = target_dimension
        new_height = max(1, round(target_dimension * height / width))
    else:
        new_height = target_dimension
        new_width = max(1, round(target_dimension * width / height))
    
    return image.resize((new_width, new_height), Image.LANCZOS).filter(ImageFilter.SHARPEN)


def create_centered_image(image, canvas_size):
    """
    Place image in the center of a white canvas.
    
    Args:
        image: PIL Image to center
        canvas_size: Size of the square canvas
    
    Returns:
        PIL Image: Image centered on white canvas
    """
    canvas = Image.new('L', (canvas_size, canvas_size), 255)
    
    img_width, img_height = image.size
    
    # Calculate centering offsets
    x_offset = (canvas_size - img_width) // 2
    y_offset = (canvas_size - img_height) // 2
    
    canvas.paste(image, (x_offset, y_offset))
    return canvas


def normalize_pixels(image):
    pixels = np.array(image.getdata(), dtype=np.float32)
    # Invert colors (white background to black) and normalize
    return (255 - pixels) / 255.0

In [7]:
image_array = load_and_preprocess_image("images/image.png")

probabilities = modele.predict(image_array)[0]
predicted_class = np.argmax(probabilities)

print(f"Selon moi l'image est : {classes[predicted_class]}\n")

for i, class_name in enumerate(classes):
    print(f"{class_name}: {probabilities[i] * 100:.2f}%")

/tmp/ipykernel_49672/94602248.py:72: DeprecationWarning: Image.Image.getdata is deprecated and will be removed in Pillow 14 (2027-10-15). Use get_flattened_data instead.
  pixels = np.array(image.getdata(), dtype=np.float32)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 160ms/step
Selon moi l'image est : Un pantalon

Un T-shirt/haut: 0.18%
Un pantalon: 99.68%
Un pull: 0.02%
Une robe: 0.03%
Un manteau: 0.01%
Des sandales: 0.01%
Une chemise: 0.01%
Des baskets: 0.00%
Un sac: 0.06%
Des bottes de cheville: 0.00%
